In [0]:

# Use your existing linked scope: retail-scope
SCOPE_NAME = "retail-scope"

# Fetch Snowflake secrets
snowflake_url = dbutils.secrets.get(
    scope=SCOPE_NAME, 
    key="snowflake-url"
)
snowflake_user = dbutils.secrets.get(
    scope=SCOPE_NAME, 
    key="snowflake-user"
)
snowflake_password = dbutils.secrets.get(
    scope=SCOPE_NAME, 
    key="snowflake-password"
)

# Snowflake connection options
sf_raw_options = {
    "sfURL": snowflake_url,
    "sfUser": snowflake_user,
    "sfPassword": snowflake_password,
    "sfDatabase": "RETAIL_CAPSTONE_DB",
    "sfSchema": "RAW",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

print("✅ Snowflake credentials loaded successfully from Azure Key Vault (retail-scope)!")


In [0]:
# ============================================================
# CELL 2: Test Snowflake Connection & Setup Database
# ============================================================

# 1. Test Snowflake connectivity
test_df = (
    spark.read
         .format("snowflake")
         .options(**sf_raw_options)
         .option("query", "SELECT CURRENT_VERSION() AS SNOWFLAKE_VERSION, CURRENT_USER() AS CONNECTED_USER, CURRENT_ROLE() AS CURRENT_ROLE")
         .load()
)
display(test_df)

# 2. Ensure database & schemas exist in Snowflake
setup_queries = [
    "CREATE DATABASE IF NOT EXISTS RETAIL_CAPSTONE_DB",
    "CREATE SCHEMA IF NOT EXISTS RETAIL_CAPSTONE_DB.RAW",
    "CREATE SCHEMA IF NOT EXISTS RETAIL_CAPSTONE_DB.GOLD"
]

for query in setup_queries:
    spark.read.format("snowflake").options(**sf_raw_options) \
        .option("query", query).load()

print("✅ RETAIL_CAPSTONE_DB and schemas (RAW, GOLD) created successfully in Snowflake!")


In [0]:
# ============================================================
# EXPORT SILVER DELTA TABLES TO SNOWFLAKE RAW (SERVERLESS COMPATIBLE)
# ============================================================

# 1. Fetch Snowflake Credentials from Azure Key Vault (retail-scope)
SCOPE_NAME = "retail-scope"

snowflake_url = dbutils.secrets.get(scope=SCOPE_NAME, key="snowflake-url")
snowflake_user = dbutils.secrets.get(scope=SCOPE_NAME, key="snowflake-user")
snowflake_password = dbutils.secrets.get(scope=SCOPE_NAME, key="snowflake-password")

# 2. Format host for Databricks Serverless (removes https:// and trailing /)
clean_host = snowflake_url.replace("https://", "").replace("http://", "").rstrip("/")

# 3. Serverless-approved connection options
sf_raw_options = {
    "host": clean_host,
    "sfuser": snowflake_user,
    "sfpassword": snowflake_password,
    "sfdatabase": "RETAIL_CAPSTONE_DB",
    "sfschema": "RAW",
    "sfwarehouse": "COMPUTE_WH",
    "sfrole": "ACCOUNTADMIN"
}

print("Snowflake connection configured for Serverless compute.")
print(f"Target: RETAIL_CAPSTONE_DB.RAW on {clean_host}")

# 4. Read Silver Delta Tables
sales_clean_df = spark.read.table("silver.sales_clean")
store_cleaned_df = spark.read.table("silver.store_cleaned")

print(f"\nLoaded silver.sales_clean:   {sales_clean_df.count():,} rows")
print(f"Loaded silver.store_cleaned: {store_cleaned_df.count():,} rows")

# 5. Write SALES_CLEAN to Snowflake RAW schema
print("\nExporting SALES_CLEAN to Snowflake (this takes ~1-2 mins for 1M rows)...")
sales_clean_df.write \
    .format("snowflake") \
    .options(**sf_raw_options) \
    .option("dbtable", "SALES_CLEAN") \
    .mode("overwrite") \
    .save()
print("✅ silver.sales_clean exported to RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN!")

# 6. Write STORE_CLEANED to Snowflake RAW schema
print("Exporting STORE_CLEANED to Snowflake...")
store_cleaned_df.write \
    .format("snowflake") \
    .options(**sf_raw_options) \
    .option("dbtable", "STORE_CLEANED") \
    .mode("overwrite") \
    .save()
print("✅ silver.store_cleaned exported to RETAIL_CAPSTONE_DB.RAW.STORE_CLEANED!")

# 7. Write STORE_CLOSURES (if table exists)
if spark.catalog.tableExists("silver.store_closures"):
    closures_df = spark.read.table("silver.store_closures")
    closures_df.write \
        .format("snowflake") \
        .options(**sf_raw_options) \
        .option("dbtable", "STORE_CLOSURES") \
        .mode("overwrite") \
        .save()
    print(f"✅ silver.store_closures exported ({closures_df.count():,} rows)!")

print("\n🎉 ALL TABLES SUCCESSFULLY EXPORTED TO SNOWFLAKE RAW SCHEMA!")
